In [14]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()

print("Project root:")
print(PROJECT_ROOT)

Project root:
c:\Users\User\OneDrive\Desktop\MAJOR PROJECT\Final_year_project


In [15]:
def find_file(filename):
    """
    Find a file inside the project directory.
    Checks common locations first, then searches subfolders.
    """
    
    common_locations = [
        PROJECT_ROOT / filename,
        PROJECT_ROOT / "data" / "raw" / filename,
        PROJECT_ROOT / "dataset" / filename,
        PROJECT_ROOT / "cleaned" / filename
    ]
    
    for path in common_locations:
        if path.exists():
            return path
    
    # Search recursively, excluding the virtual environment
    for path in PROJECT_ROOT.rglob(filename):
        if ".venv" not in path.parts:
            return path
    
    raise FileNotFoundError(
        f"Could not find '{filename}' inside {PROJECT_ROOT}"
    )


actions_path = find_file("recommendation_actions.csv")
mapping_path = find_file("context_action_mapping.csv")
bandit_train_path = find_file("train_bandit_interactions.csv")
bandit_test_path = find_file("test_bandit_interactions.csv")
train_context_path = find_file("train_business_context_cleaned.csv")
test_context_path = find_file("test_business_context_cleaned.csv")

print("Files found:")
print("Actions:", actions_path)
print("Mapping:", mapping_path)
print("Bandit train:", bandit_train_path)
print("Bandit test:", bandit_test_path)
print("Train context:", train_context_path)
print("Test context:", test_context_path)

Files found:
Actions: c:\Users\User\OneDrive\Desktop\MAJOR PROJECT\Final_year_project\data\raw\recommendation_actions.csv
Mapping: c:\Users\User\OneDrive\Desktop\MAJOR PROJECT\Final_year_project\data\raw\context_action_mapping.csv
Bandit train: c:\Users\User\OneDrive\Desktop\MAJOR PROJECT\Final_year_project\data\raw\train_bandit_interactions.csv
Bandit test: c:\Users\User\OneDrive\Desktop\MAJOR PROJECT\Final_year_project\data\raw\test_bandit_interactions.csv
Train context: c:\Users\User\OneDrive\Desktop\MAJOR PROJECT\Final_year_project\cleaned\train_business_context_cleaned.csv
Test context: c:\Users\User\OneDrive\Desktop\MAJOR PROJECT\Final_year_project\cleaned\test_business_context_cleaned.csv


In [16]:
actions = pd.read_csv(actions_path)
mapping = pd.read_csv(mapping_path)

bandit_train = pd.read_csv(bandit_train_path)
bandit_test = pd.read_csv(bandit_test_path)

train_context = pd.read_csv(train_context_path)
test_context = pd.read_csv(test_context_path)

print("All datasets loaded successfully.")

All datasets loaded successfully.


In [17]:
print("DATASET SHAPES")
print("-" * 40)

print("Actions:", actions.shape)
print("Mapping:", mapping.shape)
print("Bandit train:", bandit_train.shape)
print("Bandit test:", bandit_test.shape)
print("Train context:", train_context.shape)
print("Test context:", test_context.shape)

DATASET SHAPES
----------------------------------------
Actions: (10, 4)
Mapping: (10, 4)
Bandit train: (63840, 17)
Bandit test: (16000, 17)
Train context: (798, 31)
Test context: (200, 31)


In [18]:
print("RECOMMENDATION ACTIONS")
display(actions.head())

print("\nColumns:")
print(actions.columns.tolist())

print("\nData types:")
actions.info()

RECOMMENDATION ACTIONS


,action_id,recommendation,category,evidence_based_trigger
0,A01,Adopt digital payments,Digital,Low digital adoption and no mobile-money use
1,A02,Improve online and customer reach,Digital/Marketing,Low customer reach or low digital adoption
2,A03,Explore appropriate business finance,Finance,Finance constraint or no bank account
3,A04,Improve cash-flow and credit management,Finance,Credit buying/selling and low financial resili...
4,A05,Strengthen management practices,Management,Low management score or no profit/loss tracking



Columns:
['action_id', 'recommendation', 'category', 'evidence_based_trigger']

Data types:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   action_id               10 non-null     object
 1   recommendation          10 non-null     object
 2   category                10 non-null     object
 3   evidence_based_trigger  10 non-null     object
dtypes: object(4)
memory usage: 448.0+ bytes


In [19]:
print("ACTION DATA VALIDATION")
print("-" * 40)

print("Missing values:")
display(actions.isnull().sum())

print("\nDuplicate rows:")
print(actions.duplicated().sum())

print("\nUnique action IDs:")
print(actions["action_id"].nunique())

print("\nAction ID counts:")
display(actions["action_id"].value_counts().sort_index())

ACTION DATA VALIDATION
----------------------------------------
Missing values:


action_id                 0
recommendation            0
category                  0
evidence_based_trigger    0
dtype: int64


Duplicate rows:
0

Unique action IDs:
10

Action ID counts:


action_id
A01    1
A02    1
A03    1
A04    1
A05    1
A06    1
A07    1
A08    1
A09    1
A10    1
Name: count, dtype: int64

In [20]:
duplicate_action_ids = actions["action_id"].duplicated().sum()

print("Duplicate action IDs:", duplicate_action_ids)

if duplicate_action_ids == 0:
    print("PASS: All action IDs are unique.")
else:
    print("WARNING: Duplicate action IDs found.")

Duplicate action IDs: 0
PASS: All action IDs are unique.


In [22]:
valid_action_ids = set(actions["action_id"])
mapping_action_ids = set(mapping["action_id"])

actions_without_mapping = valid_action_ids - mapping_action_ids
invalid_mapping_actions = mapping_action_ids - valid_action_ids

print("Actions without a mapping:")
print(actions_without_mapping)

print("\nInvalid action IDs in mapping:")
print(invalid_mapping_actions)

print("\nDuplicate action IDs in mapping:")
print(mapping["action_id"].duplicated().sum())

Actions without a mapping:
set()

Invalid action IDs in mapping:
set()

Duplicate action IDs in mapping:
0


In [23]:
print("BANDIT TRAIN DATA")
print("-" * 40)

display(bandit_train.head())

print("\nColumns:")
print(bandit_train.columns.tolist())

print("\nDataset information:")
bandit_train.info()

BANDIT TRAIN DATA
----------------------------------------


,interaction_id,round,business_id,city,sector,employee_count,digital_adoption_score,finance_constraint,customer_reach_constraint,management_constraint,profit_last_month,action_id,context_action_fit,accepted,simulated_outcome,roi_proxy,reward
0,I000001,1,V2V0001,Hyderabad,Other Services,2,0.5,1,1,0,1,A01,0.35,1,0.175,0.247,0.445
1,I000002,1,V2V0001,Hyderabad,Other Services,2,0.5,1,1,0,1,A02,1.00,1,0.655,0.757,0.860
2,I000003,1,V2V0001,Hyderabad,Other Services,2,0.5,1,1,0,1,A03,1.00,1,0.991,1.000,0.997
3,I000004,1,V2V0001,Hyderabad,Other Services,2,0.5,1,1,0,1,A04,0.85,1,0.966,0.910,0.931
4,I000005,1,V2V0001,Hyderabad,Other Services,2,0.5,1,1,0,1,A05,0.40,0,0.650,0.749,0.427



Columns:
['interaction_id', 'round', 'business_id', 'city', 'sector', 'employee_count', 'digital_adoption_score', 'finance_constraint', 'customer_reach_constraint', 'management_constraint', 'profit_last_month', 'action_id', 'context_action_fit', 'accepted', 'simulated_outcome', 'roi_proxy', 'reward']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 63840 entries, 0 to 63839
Data columns (total 17 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   interaction_id             63840 non-null  object 
 1   round                      63840 non-null  int64  
 2   business_id                63840 non-null  object 
 3   city                       63840 non-null  object 
 4   sector                     63840 non-null  object 
 5   employee_count             63840 non-null  int64  
 6   digital_adoption_score     63840 non-null  float64
 7   finance_constraint         63840 non-null  int64  
 8   c

In [24]:
print("BANDIT TEST DATA")
print("-" * 40)

display(bandit_test.head())

print("\nColumns:")
print(bandit_test.columns.tolist())

print("\nDataset information:")
bandit_test.info()

BANDIT TEST DATA
----------------------------------------


,interaction_id,round,business_id,city,sector,employee_count,digital_adoption_score,finance_constraint,customer_reach_constraint,management_constraint,profit_last_month,action_id,context_action_fit,accepted,simulated_outcome,roi_proxy,reward
0,I000081,1,V2V0002,Hyderabad,Other Services,1,0.2,0,1,0,0,A01,1.00,1,0.757,0.823,0.901
1,I000082,1,V2V0002,Hyderabad,Other Services,1,0.2,0,1,0,0,A02,1.00,1,0.916,0.919,0.963
2,I000083,1,V2V0002,Hyderabad,Other Services,1,0.2,0,1,0,0,A03,0.40,0,0.407,0.374,0.298
3,I000084,1,V2V0002,Hyderabad,Other Services,1,0.2,0,1,0,0,A04,0.35,0,0.500,0.548,0.337
4,I000085,1,V2V0002,Hyderabad,Other Services,1,0.2,0,1,0,0,A05,0.40,1,0.383,0.405,0.546



Columns:
['interaction_id', 'round', 'business_id', 'city', 'sector', 'employee_count', 'digital_adoption_score', 'finance_constraint', 'customer_reach_constraint', 'management_constraint', 'profit_last_month', 'action_id', 'context_action_fit', 'accepted', 'simulated_outcome', 'roi_proxy', 'reward']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16000 entries, 0 to 15999
Data columns (total 17 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   interaction_id             16000 non-null  object 
 1   round                      16000 non-null  int64  
 2   business_id                16000 non-null  object 
 3   city                       16000 non-null  object 
 4   sector                     16000 non-null  object 
 5   employee_count             16000 non-null  int64  
 6   digital_adoption_score     16000 non-null  float64
 7   finance_constraint         16000 non-null  int64  
 8   c

In [25]:
print("TRAIN MISSING VALUES")
print("-" * 40)

display(bandit_train.isnull().sum())

print("\nTEST MISSING VALUES")
print("-" * 40)

display(bandit_test.isnull().sum())

print("\nTOTAL MISSING CELLS")

train_missing = bandit_train.isnull().sum().sum()
test_missing = bandit_test.isnull().sum().sum()

print("Train:", train_missing)
print("Test :", test_missing)

TRAIN MISSING VALUES
----------------------------------------


interaction_id               0
round                        0
business_id                  0
city                         0
sector                       0
employee_count               0
digital_adoption_score       0
finance_constraint           0
customer_reach_constraint    0
management_constraint        0
profit_last_month            0
action_id                    0
context_action_fit           0
accepted                     0
simulated_outcome            0
roi_proxy                    0
reward                       0
dtype: int64


TEST MISSING VALUES
----------------------------------------


interaction_id               0
round                        0
business_id                  0
city                         0
sector                       0
employee_count               0
digital_adoption_score       0
finance_constraint           0
customer_reach_constraint    0
management_constraint        0
profit_last_month            0
action_id                    0
context_action_fit           0
accepted                     0
simulated_outcome            0
roi_proxy                    0
reward                       0
dtype: int64


TOTAL MISSING CELLS
Train: 0
Test : 0


In [26]:
print("DUPLICATE VALIDATION")
print("-" * 40)

print("Train duplicate rows:",
      bandit_train.duplicated().sum())

print("Test duplicate rows:",
      bandit_test.duplicated().sum())

print("Train duplicate interaction IDs:",
      bandit_train["interaction_id"].duplicated().sum())

print("Test duplicate interaction IDs:",
      bandit_test["interaction_id"].duplicated().sum())

DUPLICATE VALIDATION
----------------------------------------
Train duplicate rows: 0
Test duplicate rows: 0
Train duplicate interaction IDs: 0
Test duplicate interaction IDs: 0


In [27]:
train_interaction_ids = set(bandit_train["interaction_id"])
test_interaction_ids = set(bandit_test["interaction_id"])

overlapping_interaction_ids = (
    train_interaction_ids & test_interaction_ids
)

print(
    "Interaction IDs appearing in both train and test:",
    len(overlapping_interaction_ids)
)

if len(overlapping_interaction_ids) == 0:
    print("PASS: Train and test interaction IDs are separate.")
else:
    print("WARNING: Some interaction IDs occur in both datasets.")

Interaction IDs appearing in both train and test: 0
PASS: Train and test interaction IDs are separate.


In [28]:
print("ROUND VALIDATION")
print("-" * 40)

print("TRAIN ROUND COUNTS")
display(
    bandit_train["round"]
    .value_counts()
    .sort_index()
)

print("\nTEST ROUND COUNTS")
display(
    bandit_test["round"]
    .value_counts()
    .sort_index()
)

print("\nNumber of train rounds:",
      bandit_train["round"].nunique())

print("Number of test rounds:",
      bandit_test["round"].nunique())

print("\nTrain round range:",
      bandit_train["round"].min(),
      "to",
      bandit_train["round"].max())

print("Test round range:",
      bandit_test["round"].min(),
      "to",
      bandit_test["round"].max())

ROUND VALIDATION
----------------------------------------
TRAIN ROUND COUNTS


round
1    7980
2    7980
3    7980
4    7980
5    7980
6    7980
7    7980
8    7980
Name: count, dtype: int64


TEST ROUND COUNTS


round
1    2000
2    2000
3    2000
4    2000
5    2000
6    2000
7    2000
8    2000
Name: count, dtype: int64


Number of train rounds: 8
Number of test rounds: 8

Train round range: 1 to 8
Test round range: 1 to 8


In [29]:
valid_actions = set(actions["action_id"])

invalid_train_actions = (
    set(bandit_train["action_id"]) - valid_actions
)

invalid_test_actions = (
    set(bandit_test["action_id"]) - valid_actions
)

print("Invalid action IDs in train:")
print(invalid_train_actions)

print("\nInvalid action IDs in test:")
print(invalid_test_actions)

if not invalid_train_actions and not invalid_test_actions:
    print("\nPASS: All interaction action IDs are valid.")
else:
    print("\nWARNING: Invalid action IDs found.")

Invalid action IDs in train:
set()

Invalid action IDs in test:
set()

PASS: All interaction action IDs are valid.


In [30]:
print("ACTION DISTRIBUTION")
print("-" * 40)

train_action_counts = (
    bandit_train["action_id"]
    .value_counts()
    .sort_index()
)

test_action_counts = (
    bandit_test["action_id"]
    .value_counts()
    .sort_index()
)

print("TRAIN:")
display(train_action_counts)

print("\nTEST:")
display(test_action_counts)

ACTION DISTRIBUTION
----------------------------------------
TRAIN:


action_id
A01    6384
A02    6384
A03    6384
A04    6384
A05    6384
A06    6384
A07    6384
A08    6384
A09    6384
A10    6384
Name: count, dtype: int64


TEST:


action_id
A01    1600
A02    1600
A03    1600
A04    1600
A05    1600
A06    1600
A07    1600
A08    1600
A09    1600
A10    1600
Name: count, dtype: int64

In [20]:
bandit_train.columns.tolist()

['interaction_id',
 'round',
 'business_id',
 'city',
 'sector',
 'employee_count',
 'digital_adoption_score',
 'finance_constraint',
 'customer_reach_constraint',
 'management_constraint',
 'profit_last_month',
 'action_id',
 'context_action_fit',
 'accepted',
 'simulated_outcome',
 'roi_proxy',
 'reward']

In [31]:
print("ACTION BALANCE")
print("-" * 40)

print("Train minimum action count:",
      train_action_counts.min())

print("Train maximum action count:",
      train_action_counts.max())

print("Test minimum action count:",
      test_action_counts.min())

print("Test maximum action count:",
      test_action_counts.max())

if train_action_counts.nunique() == 1:
    print("\nPASS: Training actions are perfectly balanced.")
else:
    print("\nWARNING: Training actions are imbalanced.")

ACTION BALANCE
----------------------------------------
Train minimum action count: 6384
Train maximum action count: 6384
Test minimum action count: 1600
Test maximum action count: 1600

PASS: Training actions are perfectly balanced.


In [32]:
binary_columns = [
    "finance_constraint",
    "customer_reach_constraint",
    "management_constraint",
    "profit_last_month",
    "accepted"
]

print("BINARY VARIABLE VALIDATION")
print("-" * 40)

for col in binary_columns:
    print(f"\n{col}")
    print("Unique values:",
          sorted(bandit_train[col].unique()))

BINARY VARIABLE VALIDATION
----------------------------------------

finance_constraint
Unique values: [0, 1]

customer_reach_constraint
Unique values: [0, 1]

management_constraint
Unique values: [0, 1]

profit_last_month
Unique values: [0, 1]

accepted
Unique values: [0, 1]


In [33]:
continuous_columns = [
    "digital_adoption_score",
    "context_action_fit",
    "simulated_outcome",
    "roi_proxy",
    "reward"
]

print("CONTINUOUS VARIABLE SUMMARY")
print("-" * 40)

display(
    bandit_train[continuous_columns]
    .describe()
    .T
)

CONTINUOUS VARIABLE SUMMARY
----------------------------------------


,count,mean,std,min,25%,50%,75%,max
digital_adoption_score,63840.0,0.344737,0.296458,0.00,0.000,0.200,0.500,1.0
context_action_fit,63840.0,0.733603,0.294603,0.30,0.350,0.850,1.000,1.0
simulated_outcome,63840.0,0.643738,0.257499,0.00,0.433,0.694,0.856,1.0
roi_proxy,63840.0,0.639193,0.266337,0.00,0.435,0.682,0.863,1.0
reward,63840.0,0.684462,0.267208,0.09,0.470,0.794,0.915,1.0


In [34]:
print("CONTINUOUS VARIABLE RANGES")
print("-" * 40)

for col in continuous_columns:
    print(
        f"{col}: "
        f"min={bandit_train[col].min():.4f}, "
        f"max={bandit_train[col].max():.4f}"
    )

CONTINUOUS VARIABLE RANGES
----------------------------------------
digital_adoption_score: min=0.0000, max=1.0000
context_action_fit: min=0.3000, max=1.0000
simulated_outcome: min=0.0000, max=1.0000
roi_proxy: min=0.0000, max=1.0000
reward: min=0.0900, max=1.0000


In [35]:
print("REWARD VALIDATION")
print("-" * 40)

print("TRAIN REWARD")
display(bandit_train["reward"].describe())

print("\nTEST REWARD")
display(bandit_test["reward"].describe())

print("\nTrain reward minimum:",
      bandit_train["reward"].min())

print("Train reward maximum:",
      bandit_train["reward"].max())

print("Test reward minimum:",
      bandit_test["reward"].min())

print("Test reward maximum:",
      bandit_test["reward"].max())

REWARD VALIDATION
----------------------------------------
TRAIN REWARD


count    63840.000000
mean         0.684462
std          0.267208
min          0.090000
25%          0.470000
50%          0.794000
75%          0.915000
max          1.000000
Name: reward, dtype: float64


TEST REWARD


count    16000.000000
mean         0.684504
std          0.266747
min          0.090000
25%          0.469000
50%          0.792000
75%          0.916000
max          1.000000
Name: reward, dtype: float64


Train reward minimum: 0.09
Train reward maximum: 1.0
Test reward minimum: 0.09
Test reward maximum: 1.0


In [36]:
train_invalid_rewards = bandit_train[
    (bandit_train["reward"] < 0) |
    (bandit_train["reward"] > 1)
]

test_invalid_rewards = bandit_test[
    (bandit_test["reward"] < 0) |
    (bandit_test["reward"] > 1)
]

print("Invalid train reward rows:",
      len(train_invalid_rewards))

print("Invalid test reward rows:",
      len(test_invalid_rewards))

if len(train_invalid_rewards) == 0 and len(test_invalid_rewards) == 0:
    print("\nPASS: All rewards are within [0, 1].")
else:
    print("\nWARNING: Rewards outside [0, 1] found.")

Invalid train reward rows: 0
Invalid test reward rows: 0

PASS: All rewards are within [0, 1].


In [37]:
reward_by_action = (
    bandit_train
    .groupby("action_id")["reward"]
    .agg(
        count="count",
        mean="mean",
        std="std",
        min="min",
        max="max"
    )
    .sort_values("mean", ascending=False)
)

display(reward_by_action)

,count,mean,std,min,max
action_id,,,,,
A02,6384,0.877517,0.138095,0.165,1.000
A09,6384,0.837049,0.147107,0.123,0.985
A03,6384,0.824915,0.203962,0.130,1.000
A06,6384,0.709824,0.289191,0.090,1.000
A05,6384,0.707083,0.265043,0.120,1.000
A01,6384,0.665864,0.288377,0.105,1.000
A08,6384,0.659116,0.204703,0.135,0.955
A07,6384,0.590702,0.242035,0.105,0.955
A04,6384,0.564934,0.243969,0.105,0.955


In [38]:
overall_acceptance = bandit_train["accepted"].mean()

print(
    "Overall acceptance rate:",
    round(overall_acceptance, 4)
)

acceptance_by_action = (
    bandit_train
    .groupby("action_id")["accepted"]
    .mean()
    .sort_values(ascending=False)
)

print("\nAcceptance rate by action:")
display(acceptance_by_action)

Overall acceptance rate: 0.7015

Acceptance rate by action:


action_id
A02    0.880796
A09    0.837719
A03    0.828477
A06    0.726817
A05    0.715226
A01    0.684837
A08    0.675439
A07    0.617638
A04    0.598214
A10    0.450031
Name: accepted, dtype: float64

In [39]:
outcome_by_action = (
    bandit_train
    .groupby("action_id")["simulated_outcome"]
    .agg(
        count="count",
        mean="mean",
        std="std",
        min="min",
        max="max"
    )
    .sort_values("mean", ascending=False)
)

print("SIMULATED OUTCOME BY ACTION")
display(outcome_by_action)

SIMULATED OUTCOME BY ACTION


,count,mean,std,min,max
action_id,,,,,
A02,6384,0.816362,0.156755,0.014,1.0
A09,6384,0.782628,0.163467,0.000,1.0
A03,6384,0.771754,0.205071,0.000,1.0
A05,6384,0.665129,0.252036,0.000,1.0
A06,6384,0.664845,0.277151,0.000,1.0
A01,6384,0.626198,0.273992,0.000,1.0
A08,6384,0.622876,0.204549,0.000,1.0
A07,6384,0.559317,0.239282,0.000,1.0
A04,6384,0.535764,0.237337,0.000,1.0


In [40]:
roi_by_action = (
    bandit_train
    .groupby("action_id")["roi_proxy"]
    .agg(
        count="count",
        mean="mean",
        std="std",
        min="min",
        max="max"
    )
    .sort_values("mean", ascending=False)
)

print("ROI PROXY BY ACTION")
display(roi_by_action)

ROI PROXY BY ACTION


,count,mean,std,min,max
action_id,,,,,
A02,6384,0.805893,0.171117,0.0,1.0
A09,6384,0.775397,0.180133,0.0,1.0
A03,6384,0.761583,0.215308,0.0,1.0
A05,6384,0.659268,0.260491,0.0,1.0
A06,6384,0.657776,0.280636,0.0,1.0
A08,6384,0.622370,0.224767,0.0,1.0
A01,6384,0.621948,0.281051,0.0,1.0
A07,6384,0.558677,0.253324,0.0,1.0
A04,6384,0.534001,0.253280,0.0,1.0


In [41]:
fit_by_action = (
    bandit_train
    .groupby("action_id")["context_action_fit"]
    .agg(
        count="count",
        mean="mean",
        std="std",
        min="min",
        max="max"
    )
    .sort_values("mean", ascending=False)
)

print("CONTEXT-ACTION FIT BY ACTION")
display(fit_by_action)

CONTEXT-ACTION FIT BY ACTION


,count,mean,std,min,max
action_id,,,,,
A02,6384,0.971742,0.121432,0.45,1.00
A09,6384,0.921742,0.121432,0.40,0.95
A03,6384,0.906767,0.217382,0.40,1.00
A06,6384,0.766667,0.330009,0.30,1.00
A05,6384,0.766165,0.292636,0.40,1.00
A01,6384,0.711654,0.322952,0.35,1.00
A08,6384,0.700125,0.193632,0.45,0.85
A07,6384,0.615664,0.249528,0.35,0.85
A04,6384,0.581830,0.249358,0.35,0.85


In [42]:
print("TRAIN CONTEXT COLUMNS:")
print(train_context.columns.tolist())

print("\nTEST CONTEXT COLUMNS:")
print(test_context.columns.tolist())

TRAIN CONTEXT COLUMNS:
['business_id', 'city', 'country', 'sector', 'female_majority_owned', 'migrant_owner', 'previously_unemployed_owner', 'primary_income_earner', 'owner_has_contract_job', 'household_size_proxy', 'employee_count', 'bank_account', 'has_loan', 'friends_family_finance', 'sells_on_credit', 'buys_on_credit', 'mobile_money', 'computer_or_tablet', 'electricity', 'digital_adoption_score', 'profit_last_month', 'household_based_business', 'non_fixed_premises', 'pl_statement', 'management_score_proxy', 'monthly_revenue_proxy_inr', 'monthly_profit_proxy_inr', 'finance_constraint', 'customer_reach_constraint', 'management_constraint', 'data_origin']

TEST CONTEXT COLUMNS:
['business_id', 'city', 'country', 'sector', 'female_majority_owned', 'migrant_owner', 'previously_unemployed_owner', 'primary_income_earner', 'owner_has_contract_job', 'household_size_proxy', 'employee_count', 'bank_account', 'has_loan', 'friends_family_finance', 'sells_on_credit', 'buys_on_credit', 'mobile_mo

In [43]:
if "business_id" not in train_context.columns:
    raise KeyError(
        "business_id was not found in train_context. "
        "Check the column name printed in the previous cell."
    )

if "business_id" not in test_context.columns:
    raise KeyError(
        "business_id was not found in test_context. "
        "Check the column name printed in the previous cell."
    )

train_context_ids = set(train_context["business_id"])
test_context_ids = set(test_context["business_id"])

train_bandit_ids = set(bandit_train["business_id"])
test_bandit_ids = set(bandit_test["business_id"])

missing_train_businesses = (
    train_bandit_ids - train_context_ids
)

missing_test_businesses = (
    test_bandit_ids - test_context_ids
)

print(
    "Train interaction business IDs missing from train context:",
    len(missing_train_businesses)
)

print(
    "Test interaction business IDs missing from test context:",
    len(missing_test_businesses)
)

if len(missing_train_businesses) == 0:
    print("\nPASS: All train interaction businesses exist in train context.")

if len(missing_test_businesses) == 0:
    print("PASS: All test interaction businesses exist in test context.")

Train interaction business IDs missing from train context: 0
Test interaction business IDs missing from test context: 0

PASS: All train interaction businesses exist in train context.
PASS: All test interaction businesses exist in test context.


In [44]:
print("BUSINESS COUNTS")
print("-" * 40)

print(
    "Unique businesses in train interactions:",
    bandit_train["business_id"].nunique()
)

print(
    "Unique businesses in test interactions:",
    bandit_test["business_id"].nunique()
)

print(
    "Unique businesses in train context:",
    train_context["business_id"].nunique()
)

print(
    "Unique businesses in test context:",
    test_context["business_id"].nunique()
)

BUSINESS COUNTS
----------------------------------------
Unique businesses in train interactions: 798
Unique businesses in test interactions: 200
Unique businesses in train context: 798
Unique businesses in test context: 200


In [45]:
num_businesses = bandit_train["business_id"].nunique()
num_actions = bandit_train["action_id"].nunique()
num_rounds = bandit_train["round"].nunique()

expected_rows = (
    num_businesses *
    num_actions *
    num_rounds
)

actual_rows = len(bandit_train)

print("Number of businesses:", num_businesses)
print("Number of actions:", num_actions)
print("Number of rounds:", num_rounds)

print("\nExpected rows:",
      expected_rows)

print("Actual rows:",
      actual_rows)

Number of businesses: 798
Number of actions: 10
Number of rounds: 8

Expected rows: 63840
Actual rows: 63840


In [46]:
duplicate_combinations = bandit_train.duplicated(
    subset=[
        "business_id",
        "action_id",
        "round"
    ]
).sum()

print(
    "Duplicate business-action-round combinations:",
    duplicate_combinations
)

if duplicate_combinations == 0:
    print("PASS: No duplicate business-action-round combinations.")
else:
    print("WARNING: Duplicate combinations found.")

Duplicate business-action-round combinations: 0
PASS: No duplicate business-action-round combinations.


In [47]:
round_summary = (
    bandit_train
    .groupby("round")
    .agg(
        interactions=("interaction_id", "count"),
        businesses=("business_id", "nunique"),
        actions=("action_id", "nunique"),
        avg_reward=("reward", "mean"),
        acceptance_rate=("accepted", "mean"),
        avg_roi_proxy=("roi_proxy", "mean")
    )
    .reset_index()
)

display(round_summary)

,round,interactions,businesses,actions,avg_reward,acceptance_rate,avg_roi_proxy
0,1,7980,798,10,0.683920,0.698496,0.639389
1,2,7980,798,10,0.684266,0.696491,0.640851
2,3,7980,798,10,0.684426,0.703885,0.638911
3,4,7980,798,10,0.684719,0.703509,0.638368
4,5,7980,798,10,0.684585,0.701003,0.639359
5,6,7980,798,10,0.684756,0.704135,0.638786
6,7,7980,798,10,0.683989,0.700376,0.637820
7,8,7980,798,10,0.685034,0.704261,0.640057


In [48]:
action_statistics = (
    bandit_train
    .groupby("action_id")
    .agg(
        interaction_count=("action_id", "count"),
        acceptance_rate=("accepted", "mean"),
        avg_reward=("reward", "mean"),
        reward_std=("reward", "std"),
        min_reward=("reward", "min"),
        max_reward=("reward", "max"),
        avg_roi_proxy=("roi_proxy", "mean"),
        avg_outcome=("simulated_outcome", "mean"),
        avg_context_fit=("context_action_fit", "mean")
    )
    .reset_index()
)

action_statistics = action_statistics.merge(
    actions[
        [
            "action_id",
            "recommendation",
            "category"
        ]
    ],
    on="action_id",
    how="left"
)

display(action_statistics)

,action_id,interaction_count,acceptance_rate,avg_reward,reward_std,min_reward,max_reward,avg_roi_proxy,avg_outcome,avg_context_fit,recommendation,category
0,A01,6384,0.684837,0.665864,0.288377,0.105,1.000,0.621948,0.626198,0.711654,Adopt digital payments,Digital
1,A02,6384,0.880796,0.877517,0.138095,0.165,1.000,0.805893,0.816362,0.971742,Improve online and customer reach,Digital/Marketing
2,A03,6384,0.828477,0.824915,0.203962,0.130,1.000,0.761583,0.771754,0.906767,Explore appropriate business finance,Finance
3,A04,6384,0.598214,0.564934,0.243969,0.105,0.955,0.534001,0.535764,0.581830,Improve cash-flow and credit management,Finance
4,A05,6384,0.715226,0.707083,0.265043,0.120,1.000,0.659268,0.665129,0.766165,Strengthen management practices,Management
5,A06,6384,0.726817,0.709824,0.289191,0.090,1.000,0.657776,0.664845,0.766667,Prepare regular profit and loss records,Management
6,A07,6384,0.617638,0.590702,0.242035,0.105,0.955,0.558677,0.559317,0.615664,Improve business infrastructure,Infrastructure
7,A08,6384,0.675439,0.659116,0.204703,0.135,0.955,0.622370,0.622876,0.700125,Diversify products or services,Growth
8,A09,6384,0.837719,0.837049,0.147107,0.123,0.985,0.775397,0.782628,0.921742,Expand customer reach through targeted marketing,Growth/Marketing
9,A10,6384,0.450031,0.407615,0.229277,0.090,0.985,0.395013,0.392507,0.393672,Reinvest profits for controlled business growth,Growth


In [49]:
OUTPUT_DIR = PROJECT_ROOT / "dataset"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

action_statistics_path = (
    OUTPUT_DIR / "action_statistics.csv"
)

action_statistics.to_csv(
    action_statistics_path,
    index=False
)

print("Saved:")
print(action_statistics_path)

Saved:
c:\Users\User\OneDrive\Desktop\MAJOR PROJECT\Final_year_project\dataset\action_statistics.csv


In [50]:
prepared_bandit_train = bandit_train[
    [
        "interaction_id",
        "round",
        "business_id",
        "action_id",
        "context_action_fit",
        "accepted",
        "simulated_outcome",
        "roi_proxy",
        "reward"
    ]
].copy()

display(prepared_bandit_train.head())

print("Prepared dataset shape:",
      prepared_bandit_train.shape)

,interaction_id,round,business_id,action_id,context_action_fit,accepted,simulated_outcome,roi_proxy,reward
0,I000001,1,V2V0001,A01,0.35,1,0.175,0.247,0.445
1,I000002,1,V2V0001,A02,1.00,1,0.655,0.757,0.860
2,I000003,1,V2V0001,A03,1.00,1,0.991,1.000,0.997
3,I000004,1,V2V0001,A04,0.85,1,0.966,0.910,0.931
4,I000005,1,V2V0001,A05,0.40,0,0.650,0.749,0.427


Prepared dataset shape: (63840, 9)


In [51]:
prepared_interactions_path = (
    OUTPUT_DIR / "prepared_bandit_interactions.csv"
)

prepared_bandit_train.to_csv(
    prepared_interactions_path,
    index=False
)

print("Saved:")
print(prepared_interactions_path)

Saved:
c:\Users\User\OneDrive\Desktop\MAJOR PROJECT\Final_year_project\dataset\prepared_bandit_interactions.csv


In [52]:
print("OUTPUT FILE VERIFICATION")
print("-" * 40)

print(
    "action_statistics.csv exists:",
    action_statistics_path.exists()
)

print(
    "prepared_bandit_interactions.csv exists:",
    prepared_interactions_path.exists()
)

if action_statistics_path.exists():
    print(
        "action_statistics.csv size:",
        action_statistics_path.stat().st_size,
        "bytes"
    )

if prepared_interactions_path.exists():
    print(
        "prepared_bandit_interactions.csv size:",
        prepared_interactions_path.stat().st_size,
        "bytes"
    )

OUTPUT FILE VERIFICATION
----------------------------------------
action_statistics.csv exists: True
prepared_bandit_interactions.csv exists: True
action_statistics.csv size: 1978 bytes
prepared_bandit_interactions.csv size: 2985085 bytes


In [53]:
print("=" * 60)
print("MEMBER 3 BANDIT DATA VALIDATION SUMMARY")
print("=" * 60)

print("\nDATASETS")
print("Actions:", actions.shape)
print("Mapping:", mapping.shape)
print("Train interactions:", bandit_train.shape)
print("Test interactions:", bandit_test.shape)

print("\nMISSING VALUES")
print("Train missing cells:", train_missing)
print("Test missing cells:", test_missing)

print("\nDUPLICATES")
print("Train duplicate rows:",
      bandit_train.duplicated().sum())
print("Test duplicate rows:",
      bandit_test.duplicated().sum())

print("\nACTIONS")
print("Number of actions:",
      actions["action_id"].nunique())

print("Invalid train action IDs:",
      len(invalid_train_actions))

print("Invalid test action IDs:",
      len(invalid_test_actions))

print("\nREWARD")
print("Train reward range:",
      bandit_train["reward"].min(),
      "to",
      bandit_train["reward"].max())

print("Test reward range:",
      bandit_test["reward"].min(),
      "to",
      bandit_test["reward"].max())

print("\nACCEPTANCE")
print("Overall acceptance rate:",
      round(overall_acceptance, 4))

print("\nSEQUENCE")
print("Train rounds:",
      bandit_train["round"].nunique())

print("Test rounds:",
      bandit_test["round"].nunique())

print("\nSTRUCTURE")
print("Expected train rows:",
      expected_rows)

print("Actual train rows:",
      actual_rows)

print(
    "Duplicate business-action-round combinations:",
    duplicate_combinations
)

print("\nOUTPUTS")
print("action_statistics.csv:",
      action_statistics_path)

print("prepared_bandit_interactions.csv:",
      prepared_interactions_path)

print("\n" + "=" * 60)
print("VALIDATION COMPLETE")
print("=" * 60)

MEMBER 3 BANDIT DATA VALIDATION SUMMARY

DATASETS
Actions: (10, 4)
Mapping: (10, 4)
Train interactions: (63840, 17)
Test interactions: (16000, 17)

MISSING VALUES
Train missing cells: 0
Test missing cells: 0

DUPLICATES
Train duplicate rows: 0
Test duplicate rows: 0

ACTIONS
Number of actions: 10
Invalid train action IDs: 0
Invalid test action IDs: 0

REWARD
Train reward range: 0.09 to 1.0
Test reward range: 0.09 to 1.0

ACCEPTANCE
Overall acceptance rate: 0.7015

SEQUENCE
Train rounds: 8
Test rounds: 8

STRUCTURE
Expected train rows: 63840
Actual train rows: 63840
Duplicate business-action-round combinations: 0

OUTPUTS
action_statistics.csv: c:\Users\User\OneDrive\Desktop\MAJOR PROJECT\Final_year_project\dataset\action_statistics.csv
prepared_bandit_interactions.csv: c:\Users\User\OneDrive\Desktop\MAJOR PROJECT\Final_year_project\dataset\prepared_bandit_interactions.csv

VALIDATION COMPLETE


In [54]:
validation_report = pd.DataFrame({
    "check": [
        "Train missing cells",
        "Test missing cells",
        "Train duplicate rows",
        "Test duplicate rows",
        "Train duplicate interaction IDs",
        "Test duplicate interaction IDs",
        "Invalid train action IDs",
        "Invalid test action IDs",
        "Train reward minimum",
        "Train reward maximum",
        "Test reward minimum",
        "Test reward maximum",
        "Duplicate business-action-round combinations"
    ],
    "result": [
        train_missing,
        test_missing,
        bandit_train.duplicated().sum(),
        bandit_test.duplicated().sum(),
        bandit_train["interaction_id"].duplicated().sum(),
        bandit_test["interaction_id"].duplicated().sum(),
        len(invalid_train_actions),
        len(invalid_test_actions),
        bandit_train["reward"].min(),
        bandit_train["reward"].max(),
        bandit_test["reward"].min(),
        bandit_test["reward"].max(),
        duplicate_combinations
    ]
})

validation_report_path = (
    OUTPUT_DIR / "bandit_validation_report.csv"
)

validation_report.to_csv(
    validation_report_path,
    index=False
)

display(validation_report)

print("Saved:")
print(validation_report_path)

,check,result
0,Train missing cells,0.00
1,Test missing cells,0.00
2,Train duplicate rows,0.00
3,Test duplicate rows,0.00
4,Train duplicate interaction IDs,0.00
5,Test duplicate interaction IDs,0.00
6,Invalid train action IDs,0.00
7,Invalid test action IDs,0.00
8,Train reward minimum,0.09
9,Train reward maximum,1.00


Saved:
c:\Users\User\OneDrive\Desktop\MAJOR PROJECT\Final_year_project\dataset\bandit_validation_report.csv


In [55]:
validation_report = pd.DataFrame({
    "check": [
        "Train missing cells",
        "Test missing cells",
        "Train duplicate rows",
        "Test duplicate rows",
        "Train duplicate interaction IDs",
        "Test duplicate interaction IDs",
        "Invalid train action IDs",
        "Invalid test action IDs",
        "Train reward minimum",
        "Train reward maximum",
        "Test reward minimum",
        "Test reward maximum",
        "Duplicate business-action-round combinations"
    ],
    "result": [
        train_missing,
        test_missing,
        bandit_train.duplicated().sum(),
        bandit_test.duplicated().sum(),
        bandit_train["interaction_id"].duplicated().sum(),
        bandit_test["interaction_id"].duplicated().sum(),
        len(invalid_train_actions),
        len(invalid_test_actions),
        bandit_train["reward"].min(),
        bandit_train["reward"].max(),
        bandit_test["reward"].min(),
        bandit_test["reward"].max(),
        duplicate_combinations
    ]
})

validation_report_path = (
    OUTPUT_DIR / "bandit_validation_report.csv"
)

validation_report.to_csv(
    validation_report_path,
    index=False
)

display(validation_report)

print("Saved:")
print(validation_report_path)

,check,result
0,Train missing cells,0.00
1,Test missing cells,0.00
2,Train duplicate rows,0.00
3,Test duplicate rows,0.00
4,Train duplicate interaction IDs,0.00
5,Test duplicate interaction IDs,0.00
6,Invalid train action IDs,0.00
7,Invalid test action IDs,0.00
8,Train reward minimum,0.09
9,Train reward maximum,1.00


Saved:
c:\Users\User\OneDrive\Desktop\MAJOR PROJECT\Final_year_project\dataset\bandit_validation_report.csv


In [56]:
print("""
IMPORTANT MODELING NOTE
-----------------------

The following variables are feedback/outcome-related:

- accepted
- simulated_outcome
- roi_proxy
- reward

These variables should NOT be included as ordinary input features
when constructing the LinUCB context vector.

context_action_fit should also be treated carefully because it
describes the relationship between a context and an action.

The final LinUCB context should be constructed from the contextual
features produced by Member 2's feature-engineering pipeline.

The prepared interaction dataset provides the action and feedback
information needed to evaluate/update the bandit.
""")


IMPORTANT MODELING NOTE
-----------------------

The following variables are feedback/outcome-related:

- accepted
- simulated_outcome
- roi_proxy
- reward

These variables should NOT be included as ordinary input features
when constructing the LinUCB context vector.

context_action_fit should also be treated carefully because it
describes the relationship between a context and an action.

The final LinUCB context should be constructed from the contextual
features produced by Member 2's feature-engineering pipeline.

The prepared interaction dataset provides the action and feedback
information needed to evaluate/update the bandit.

